# Solutions · 04 · The heat equation in every form

Worked solutions to the exercises of [notebook 04](../notebooks/04_the_heat_equation_in_every_form.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engmolines is missing): install it from GitHub
    import engmolines
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engmolines"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engmolines import problems as pr, integrators as ti, grids, boundary as bc, operators as op
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

## Exercise 1

The periodically heated slab has penetration depth 1/k with k = sqrt(omega / 2 kappa). Solve it on a domain of length 2/k and of length 8/k with the far boundary insulated instead of imposed; how does the far-boundary choice affect the solution near x = 0, and at what domain length does it stop mattering?

In [2]:
kappa, omega = 1.0, 2.0
k = np.sqrt(omega / (2 * kappa))
ex = lambda x, t: np.exp(-k * x) * np.cos(omega * t - k * x)      # noqa: E731
for L_over in (2, 8):
    L = L_over / k
    g = grids.uniform(0, L, 40 * L_over // 2)
    f = lambda t, u, g=g: op.diffusion(u, g, kappa, bc.Dirichlet(lambda t: ex(0.0, t)), bc.Neumann(0.0), t)
    s = ti.bdf(f, ex(g.x, 0.0), (0, 3 * np.pi), 0.01, 2, band=(1, 1), n_out=1)
    near = g.x < 1 / k
    print(f"domain {L_over}/k with an insulated far end: error near x = 0 (x < 1/k) {np.max(np.abs(s.y[-1][near] - ex(g.x[near], s.t[-1]))):.2e}; at the far end {abs(s.y[-1][-1] - ex(L, s.t[-1])):.2e}")

domain 2/k with an insulated far end: error near x = 0 (x < 1/k) 4.93e-02; at the far end 6.06e-02


domain 8/k with an insulated far end: error near x = 0 (x < 1/k) 6.80e-05; at the far end 2.34e-06


With the far boundary at 2/k the insulated wall reflects the thermal wave and the error near the surface is a
few percent; at 8/k the wave has decayed by e⁻⁸ before reaching it and the choice of far condition no longer
matters. The rule: put artificial boundaries several penetration depths away, or impose the decaying solution
there.

## Exercise 2

Make the composite wall's second layer ten times more conductive and a tenth as thick (equal resistance). Confirm the same steady flux, and compare the time to reach 95 % of it.

In [3]:
for name, k2, L2 in (("original", 5.0, 0.5), ("ten times more conductive, a tenth as thick", 50.0, 0.05)):
    g = grids.cell_centred(0, 0.5 + L2, 200)
    kk = np.where(g.x < 0.5, 1.0, k2)
    f = lambda t, u, g=g, kk=kk: op.diffusion(u, g, kk, bc.Dirichlet(1.0), bc.Dirichlet(0.0), t)
    s = ti.bdf(f, np.zeros(200), (0, 3.0), 0.01, 2, band=(1, 1), n_out=60)
    q = [(1.0 - y[0]) / (g.dx / 2) * 1.0 for y in s.y]              # flux at the left face from the first cell
    q_ss = (1 - 0) / (0.5 / 1.0 + L2 / k2)
    t95 = s.t[np.argmax(np.array(q) <= 1.05 * q_ss)]
    print(f"{name:<44}: steady flux {q[-1]:.4f} (series resistances {q_ss:.4f}); within 5 % after t = {t95:.2f}")

original                                    : steady flux 1.6667 (series resistances 1.6667); within 5 % after t = 0.15


ten times more conductive, a tenth as thick : steady flux 1.9941 (series resistances 1.9960); within 5 % after t = 0.10


Equal resistance means an identical steady flux, but the thin conductive layer holds far less heat (its
thermal mass is a tenth), so the wall reaches its steady state sooner: the transient is governed by the
capacities, the steady state by the resistances.

## Exercise 3

**Implement it yourself:** write the cylindrical diffusion operator with the control-volume treatment at r = 0 and reproduce the cooling-cylinder decay rate.

In [4]:
def cyl_rhs(u, r, kappa):
    dr = r[1] - r[0]; n = len(u)
    out = np.zeros(n)
    rf = np.r_[0.0, 0.5 * (r[:-1] + r[1:])]                      # inner face radius of each control volume
    rr = np.r_[0.5 * (r[:-1] + r[1:]), r[-1] + dr / 2]            # outer face radius
    flux = np.r_[0.0, kappa * np.diff(u) / dr, 0.0]               # k du/dr at the faces (zero at the axis; outer handled by Dirichlet below)
    vol = (rr**2 - rf**2) / 2
    out[:-1] = (rr[:-1] * flux[1:-1] - rf[:-1] * flux[:-2]) / vol[:-1]
    return out                                                     # last node fixed at 0: Dirichlet
from scipy.special import j0
p = pr.cooling_cylinder(n=60); r = p.x; u0 = j0(2.404825557695773 * r)
s = ti.bdf(lambda t, u: cyl_rhs(u, r, 1.0), u0, (0, 0.2), 1e-3, 4, band=(1, 1), n_out=1)
print(f"hand-written cylindrical operator: error vs exp(-j01^2 t) J0 = {np.max(np.abs(s.y[-1] - np.exp(-2.404825557695773**2 * 0.2) * u0)):.1e}")

hand-written cylindrical operator: error vs exp(-j01^2 t) J0 = 5.6e-05


The control-volume form - fluxes through faces weighted by face radius, divided by the volume between faces -
has no 1/r to divide by at the axis: the innermost volume is a disc of radius Δr/2 with a single outer face.
It reproduces the exact Bessel decay rate like the library's operator.